<a href="https://colab.research.google.com/github/EstibenMT/machine_learning/blob/main/E_2/Taller2_Limpieza_y_Calidad_de_Datos_con_Pandas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Taller 2: Limpieza y calidad de datos con Pandas

## Análisis y depuración de un conjunto de datos de ventas

**Integrantes:** Escribir aquí los nombres del equipo  
**Asignatura:** Aprendizaje Computacional  
**Actividad:** Taller 2  
**Herramienta utilizada:** Google Colab

## Integrantes

- Estiben Montoya Taborda
- Luis David Lopera Parra
- Santiago Gomez Gallego
- Jose Miguel Buritica Morales

## Introducción

Como equipo de trabajo realizamos un proceso de revisión y limpieza sobre un conjunto de datos relacionado con ventas de clientes.

Antes de utilizar cualquier información para análisis o modelos de aprendizaje computacional, es importante verificar que los datos estén completos, que no existan registros repetidos y que las columnas realmente aporten información útil.

En este taller usamos la librería Pandas para cargar, revisar, analizar y corregir los problemas encontrados en el archivo `datos_ventas_estudiantes.csv`.

La idea no es solamente ejecutar código, sino entender qué problema tiene cada columna, tomar una decisión y dejar evidencia de por qué se realizó cada cambio.

## Descripción de las variables

| Columna | Descripción | Tipo de dato esperado | Decisión |
|---|---|---|---|
| `ID_Cliente` | Identificador del cliente | Texto o entero | Conservar |
| `Pais` | País donde se realizó la venta | Texto | Eliminar porque todos los registros tienen el mismo país |
| `Ciudad` | Ciudad del cliente | Texto | Conservar |
| `Vendedor` | Persona encargada de la venta | Texto | Conservar |
| `Categoria` | Categoría del producto vendido | Texto | Conservar y completar faltantes |
| `Edad_Cliente` | Edad del cliente | Numérico | Conservar y completar faltantes |
| `Monto_Venta` | Valor de la venta | Numérico | Conservar y completar faltantes |
| `Comentarios_Cliente` | Comentario realizado por el cliente | Texto | Eliminar por exceso de valores faltantes |

Conservar el identificador nos permite mantener la trazabilidad de los registros. Sin embargo, si más adelante se construye un modelo predictivo, esta columna no debería utilizarse como variable de entrada porque solamente identifica al cliente.

In [3]:
# Importamos Pandas, que será la librería principal para trabajar con los datos.
import pandas as pd

# Esta función permite seleccionar y cargar el archivo desde nuestro computador.
from google.colab import files

# Abrimos la ventana para subir el archivo CSV.
uploaded_files = files.upload()

# Obtenemos automáticamente el nombre del archivo cargado.
file_name = next(iter(uploaded_files))

# Leemos el archivo CSV y lo guardamos en un DataFrame.
original_df = pd.read_csv(file_name)

# Mostramos un mensaje para confirmar que el archivo fue cargado.
print(f"Archivo cargado correctamente: {file_name}")

Saving datos_ventas_estudiantes.csv to datos_ventas_estudiantes (1).csv
Archivo cargado correctamente: datos_ventas_estudiantes (1).csv


En esta parte cargamos el archivo entregado para el taller. Utilizamos `read_csv()` porque la información se encuentra almacenada en formato CSV.

El resultado se guarda en `original_df`, que representa la copia original de los datos. Conservamos este DataFrame sin modificar para poder comparar el estado inicial con el resultado final de la limpieza.

## Paso 1. Diagnóstico inicial

Primero revisamos cómo está compuesto el conjunto de datos antes de realizar cualquier modificación.

En esta etapa nos interesa conocer:

- Cuántas filas y columnas existen.
- Cuáles son los nombres de las columnas.
- Qué tipo de dato tiene cada variable.
- Cuánta información está disponible.

In [4]:
# Mostramos la cantidad de filas y columnas.
print("Dimensiones del conjunto de datos:")
print(original_df.shape)

# Mostramos los nombres de las columnas.
print("\nNombres de las columnas:")
print(original_df.columns.tolist())

# Revisamos tipos de datos y cantidad de valores no nulos.
print("\nInformación general:")
original_df.info()

Dimensiones del conjunto de datos:
(1590, 8)

Nombres de las columnas:
['ID_Cliente', 'Pais', 'Ciudad', 'Vendedor', 'Categoria', 'Edad_Cliente', 'Monto_Venta', 'Comentarios_Cliente']

Información general:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1590 entries, 0 to 1589
Data columns (total 8 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   ID_Cliente           1590 non-null   object 
 1   Pais                 1590 non-null   object 
 2   Ciudad               1590 non-null   object 
 3   Vendedor             1590 non-null   object 
 4   Categoria            1419 non-null   object 
 5   Edad_Cliente         1390 non-null   float64
 6   Monto_Venta          1458 non-null   float64
 7   Comentarios_Cliente  227 non-null    object 
dtypes: float64(2), object(6)
memory usage: 99.5+ KB


El conjunto original contiene 1.590 filas y 8 columnas.

La función `info()` nos ayuda a confirmar que las columnas numéricas son `Edad_Cliente` y `Monto_Venta`, mientras que las demás corresponden a información categórica o de texto.

También podemos observar que algunas columnas tienen menos valores no nulos que el total de filas. Esto nos indica que existen datos faltantes.

In [5]:
# Mostramos los primeros diez registros para observar el formato real de los datos.
original_df.head(10)

,ID_Cliente,Pais,Ciudad,Vendedor,Categoria,Edad_Cliente,Monto_Venta,Comentarios_Cliente
0,CLI-2079,Colombia,Cali,Laura Torres,Ropa,56.0,8575896.0,NaN
1,CLI-1405,Colombia,Cali,Laura Torres,Ropa,23.0,9635301.0,NaN
2,CLI-2493,Colombia,Bogotá,Carlos Ruiz,Ropa,60.0,73060.0,Excelente servicio
3,CLI-1239,Colombia,Medellín,Carlos Ruiz,Electrónica,36.0,232758.0,NaN
4,CLI-1610,Colombia,Cali,Laura Torres,Hogar,26.0,677935.0,NaN
5,CLI-1099,Colombia,Barranquilla,Laura Torres,Hogar,61.0,5184986.0,NaN
6,CLI-2145,Colombia,Barranquilla,Laura Torres,Deportes,21.0,7604525.0,NaN
7,CLI-1297,Colombia,Barranquilla,Carlos Ruiz,Hogar,58.0,4062451.0,NaN
8,CLI-1932,Colombia,Medellín,María Gómez,Deportes,51.0,5885467.0,Excelente servicio
9,CLI-1275,Colombia,Bogotá,Laura Torres,Hogar,20.0,4834993.0,NaN


Con `head()` observamos una muestra de los registros. Esto permite revisar si los valores tienen sentido, si las columnas están bien separadas y si existen datos vacíos visibles.

Esta revisión inicial es importante porque los datos no se deben analizar únicamente desde los nombres de las columnas. También debemos observar cómo vienen escritos los valores.

In [6]:
# Generamos un resumen estadístico de las columnas numéricas y categóricas.
original_df.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
ID_Cliente,1590,1500,CLI-1451,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Pais,1590,1,Colombia,1590,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Ciudad,1590,4,Barranquilla,413,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Vendedor,1590,5,Juan Pérez,337,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Categoria,1419,4,Electrónica,362,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Edad_Cliente,1390.0,NaN,NaN,NaN,41.369065,13.454766,18.0,30.0,42.0,53.0,64.0
Monto_Venta,1458.0,NaN,NaN,NaN,4997349.243484,2885749.828109,51875.0,2449505.25,5097124.5,7489700.25,9997152.0
Comentarios_Cliente,227,1,Excelente servicio,227,NaN,NaN,NaN,NaN,NaN,NaN,NaN


El resumen estadístico nos permite tener una visión rápida del comportamiento de las variables.

Para la edad encontramos valores entre 18 y 64 años, lo cual no muestra edades imposibles. En el monto de venta también observamos valores positivos.

La columna `Comentarios_Cliente` tiene muy pocos valores registrados, por lo que posteriormente evaluamos si realmente conviene conservarla.

### Identificación de valores faltantes

Ahora calculamos los valores nulos por columna. No basta con conocer la cantidad total; también calculamos el porcentaje para saber qué tan afectada está cada variable.

Este porcentaje nos ayuda a tomar decisiones más objetivas sobre conservar, eliminar o completar una columna.

In [7]:
# Contamos los valores faltantes de cada columna.
missing_count = original_df.isna().sum()

# Calculamos el porcentaje de valores faltantes.
missing_percentage = (missing_count / len(original_df) * 100).round(2)

# Construimos una tabla resumen.
missing_summary = pd.DataFrame({
    "missing_count": missing_count,
    "missing_percentage": missing_percentage
})

# Ordenamos de mayor a menor cantidad de valores faltantes.
missing_summary = missing_summary.sort_values(
    by="missing_count",
    ascending=False
)

missing_summary

,missing_count,missing_percentage
Comentarios_Cliente,1363,85.72
Edad_Cliente,200,12.58
Categoria,171,10.75
Monto_Venta,132,8.30
Vendedor,0,0.00
Ciudad,0,0.00
Pais,0,0.00
ID_Cliente,0,0.00


Los resultados muestran lo siguiente:

- `Comentarios_Cliente` tiene 1.363 valores faltantes, equivalentes al 85,72 %.
- `Categoria` tiene 171 valores faltantes.
- `Edad_Cliente` tiene 200 valores faltantes.
- `Monto_Venta` tiene 132 valores faltantes.

La columna de comentarios está demasiado incompleta. Además, los pocos comentarios registrados contienen prácticamente el mismo texto, por lo que completarla podría generar información artificial.

In [8]:
# Revisamos las columnas de texto para identificar valores que puedan representar
# vacíos escritos de otra forma, como "NA", "N/A", "null" o cadenas vacías.
text_columns = original_df.select_dtypes(include="object").columns

suspicious_values = ["", "NA", "N/A", "null", "None", "nan"]
suspicious_records = []

for column in text_columns:
    normalized_values = original_df[column].astype("string").str.strip()

    for suspicious_value in suspicious_values:
        count = (normalized_values == suspicious_value).sum()

        if count > 0:
            suspicious_records.append({
                "column": column,
                "value": suspicious_value,
                "count": int(count)
            })

suspicious_records_df = pd.DataFrame(suspicious_records)

if suspicious_records_df.empty:
    print("No se encontraron valores sospechosos escritos como texto.")
else:
    suspicious_records_df

No se encontraron valores sospechosos escritos como texto.


Además de los valores nulos reconocidos por Pandas, revisamos si existían textos como `NA`, `null` o espacios vacíos.

En este caso no se encontraron valores sospechosos adicionales. Por esta razón, trabajamos directamente con los valores identificados mediante `isna()`.

In [9]:
# Identificamos las columnas categóricas.
categorical_columns = original_df.select_dtypes(include="object").columns

# Mostramos los valores más frecuentes de cada columna categórica.
for column in categorical_columns:
    print(f"\nFrecuencia de valores en: {column}")
    print(original_df[column].value_counts(dropna=False))


Frecuencia de valores en: ID_Cliente
ID_Cliente
CLI-1451    2
CLI-1067    2
CLI-1289    2
CLI-1924    2
CLI-2281    2
           ..
CLI-1722    1
CLI-1682    1
CLI-1477    1
CLI-1714    1
CLI-2312    1
Name: count, Length: 1500, dtype: int64

Frecuencia de valores en: Pais
Pais
Colombia    1590
Name: count, dtype: int64

Frecuencia de valores en: Ciudad
Ciudad
Barranquilla    413
Cali            406
Bogotá          401
Medellín        370
Name: count, dtype: int64

Frecuencia de valores en: Vendedor
Vendedor
Juan Pérez      337
Laura Torres    334
Carlos Ruiz     318
María Gómez     316
Ana López       285
Name: count, dtype: int64

Frecuencia de valores en: Categoria
Categoria
Electrónica    362
Ropa           360
Hogar          349
Deportes       348
NaN            171
Name: count, dtype: int64

Frecuencia de valores en: Comentarios_Cliente
Comentarios_Cliente
NaN                   1363
Excelente servicio     227
Name: count, dtype: int64


Esta revisión nos permite saber si las variables categóricas tienen variedad o si contienen un único valor.

Observamos que `Pais` solamente contiene el valor `Colombia`. Como no existe variación, esta columna no ayuda a diferenciar los registros y se puede eliminar.

También observamos que `Comentarios_Cliente` tiene pocos comentarios registrados y la mayoría corresponden a valores faltantes.

### Identificación de registros duplicados

Un registro duplicado es una fila que aparece más de una vez con exactamente los mismos valores en todas sus columnas.

Los duplicados pueden alterar conteos, promedios y resultados estadísticos. Por eso los identificamos antes de terminar la limpieza.

In [10]:
# Contamos cuántas filas están repetidas completamente.
duplicate_count = original_df.duplicated().sum()

print(f"Cantidad de filas duplicadas: {duplicate_count}")

# Mostramos algunos ejemplos de filas duplicadas.
duplicated_rows = original_df[
    original_df.duplicated(keep=False)
].sort_values(by="ID_Cliente")

duplicated_rows.head(10)

Cantidad de filas duplicadas: 90


,ID_Cliente,Pais,Ciudad,Vendedor,Categoria,Edad_Cliente,Monto_Venta,Comentarios_Cliente
219,CLI-1029,Colombia,Bogotá,Juan Pérez,Hogar,44.0,3005179.0,NaN
24,CLI-1029,Colombia,Bogotá,Juan Pérez,Hogar,44.0,3005179.0,NaN
92,CLI-1049,Colombia,Bogotá,Ana López,Hogar,37.0,NaN,NaN
656,CLI-1049,Colombia,Bogotá,Ana López,Hogar,37.0,NaN,NaN
120,CLI-1051,Colombia,Cali,Laura Torres,Hogar,64.0,8386366.0,NaN
415,CLI-1051,Colombia,Cali,Laura Torres,Hogar,64.0,8386366.0,NaN
60,CLI-1065,Colombia,Cali,María Gómez,NaN,21.0,5274063.0,NaN
547,CLI-1065,Colombia,Cali,María Gómez,NaN,21.0,5274063.0,NaN
139,CLI-1067,Colombia,Cali,Laura Torres,Hogar,35.0,5176717.0,NaN
679,CLI-1067,Colombia,Cali,Laura Torres,Hogar,35.0,5176717.0,NaN


Se encontraron 90 filas completamente duplicadas.

Decidimos conservar solamente la primera aparición de cada registro y eliminar las repeticiones. Esta decisión evita contar dos veces la misma venta y permite que cada fila represente un registro único.

## Paso 2. Creación de una copia para trabajar

No modificamos directamente el DataFrame original. Creamos una copia llamada `working_df`.

Esta práctica permite conservar los datos iniciales y comparar posteriormente el antes y el después de la limpieza.

In [11]:
# Creamos una copia independiente del DataFrame original.
working_df = original_df.copy()

print("Copia de trabajo creada correctamente.")

Copia de trabajo creada correctamente.


## Paso 3. Selección de columnas útiles

Como equipo analizamos cada variable según su utilidad para el estudio.

Decidimos eliminar:

- `Pais`: todos los registros tienen el mismo valor, `Colombia`.
- `Comentarios_Cliente`: tiene 85,72 % de datos faltantes y muy poca variedad.

No eliminamos filas solamente porque una columna tenga datos faltantes. En su lugar, conservamos las variables importantes y luego completamos sus valores vacíos utilizando estadísticas adecuadas.

In [12]:
# Definimos las columnas que no aportan suficiente información.
columns_to_drop = [
    "Pais",
    "Comentarios_Cliente"
]

# Eliminamos las columnas seleccionadas.
clean_df = working_df.drop(
    columns=columns_to_drop
).copy()

print("Columnas eliminadas:")
print(columns_to_drop)

print("\nColumnas restantes:")
print(clean_df.columns.tolist())

Columnas eliminadas:
['Pais', 'Comentarios_Cliente']

Columnas restantes:
['ID_Cliente', 'Ciudad', 'Vendedor', 'Categoria', 'Edad_Cliente', 'Monto_Venta']


In [ ]:
# Revisamos nuevamente los valores faltantes después de eliminar
# las columnas que decidimos retirar.
clean_df.isna().sum()

Después de eliminar las columnas `Pais` y `Comentarios_Cliente`, permanecen valores faltantes en `Categoria`, `Edad_Cliente` y `Monto_Venta`.

Estas columnas sí son importantes para el análisis, por lo que no las eliminamos. En el siguiente paso las completamos con métodos estadísticos.

### Tratamiento de valores faltantes en `Edad_Cliente`

Para completar las edades utilizamos la mediana.

La mediana es el valor central de los datos ordenados. La escogimos porque es menos sensible que el promedio cuando existen valores muy altos o muy bajos.

De esta manera no eliminamos clientes y tampoco alteramos demasiado la distribución original de las edades.

In [13]:
# Calculamos la mediana de la edad utilizando únicamente los valores disponibles.
median_age = clean_df["Edad_Cliente"].median()

# Completamos los valores faltantes con la mediana calculada.
clean_df["Edad_Cliente"] = clean_df["Edad_Cliente"].fillna(median_age)

print(f"Mediana utilizada para Edad_Cliente: {median_age}")
print(f"Valores faltantes restantes: {clean_df['Edad_Cliente'].isna().sum()}")

Mediana utilizada para Edad_Cliente: 42.0
Valores faltantes restantes: 0


### Tratamiento de valores faltantes en `Monto_Venta`

Para el monto de venta también utilizamos la mediana.

Los valores de venta pueden variar bastante entre clientes. Por eso preferimos la mediana en lugar del promedio, ya que representa mejor un valor típico y evita que los valores extremos influyan demasiado.

In [14]:
# Calculamos la mediana de los montos de venta existentes.
median_sales_amount = clean_df["Monto_Venta"].median()

# Completamos los montos faltantes.
clean_df["Monto_Venta"] = clean_df["Monto_Venta"].fillna(
    median_sales_amount
)

print(f"Mediana utilizada para Monto_Venta: {median_sales_amount}")
print(f"Valores faltantes restantes: {clean_df['Monto_Venta'].isna().sum()}")

Mediana utilizada para Monto_Venta: 5097124.5
Valores faltantes restantes: 0


### Tratamiento de valores faltantes en `Categoria`

La columna `Categoria` es una variable categórica. Para completarla utilizamos la moda, es decir, la categoría que aparece con mayor frecuencia.

No tendría sentido utilizar una media porque las categorías son textos. La moda mantiene el formato original de la columna.

In [15]:
# Obtenemos la categoría más frecuente ignorando los valores faltantes.
category_mode = clean_df["Categoria"].mode(dropna=True).iloc[0]

# Completamos los valores faltantes con la categoría más frecuente.
clean_df["Categoria"] = clean_df["Categoria"].fillna(category_mode)

print(f"Categoría utilizada para completar faltantes: {category_mode}")
print(f"Valores faltantes restantes: {clean_df['Categoria'].isna().sum()}")

Categoría utilizada para completar faltantes: Electrónica
Valores faltantes restantes: 0


In [16]:
# Revisamos los valores faltantes de todas las columnas.
remaining_missing = clean_df.isna().sum()

print("Valores faltantes por columna:")
print(remaining_missing)

print(f"\nTotal de valores faltantes: {remaining_missing.sum()}")

Valores faltantes por columna:
ID_Cliente      0
Ciudad          0
Vendedor        0
Categoria       0
Edad_Cliente    0
Monto_Venta     0
dtype: int64

Total de valores faltantes: 0


Después de aplicar las estrategias de imputación, todas las columnas que decidimos conservar quedaron completas.

Usamos la mediana para las variables numéricas y la moda para la variable categórica. Esta decisión conserva la mayor cantidad posible de registros.

## Paso 4. Tratamiento de registros duplicados

Ya identificamos los duplicados durante el diagnóstico inicial. Ahora eliminamos las filas repetidas.

Usamos `keep="first"` para conservar la primera aparición y eliminar las copias posteriores. Esto permite mantener un registro representativo sin perder completamente la información.

In [17]:
# Guardamos la cantidad de duplicados antes de eliminarlos.
duplicates_before_cleaning = clean_df.duplicated().sum()

# Eliminamos las filas duplicadas y reorganizamos el índice.
clean_df = clean_df.drop_duplicates(
    keep="first"
).reset_index(drop=True)

# Verificamos la cantidad de duplicados después de la limpieza.
duplicates_after_cleaning = clean_df.duplicated().sum()

print(f"Duplicados antes de la limpieza: {duplicates_before_cleaning}")
print(f"Duplicados después de la limpieza: {duplicates_after_cleaning}")

Duplicados antes de la limpieza: 90
Duplicados después de la limpieza: 0


## Validación final del conjunto limpio

En esta etapa comprobamos que el proceso haya cumplido su objetivo.

Verificamos:

- Cantidad final de filas y columnas.
- Valores faltantes.
- Duplicados.
- Nombres de las columnas.
- Tipos de datos.

In [18]:
# Calculamos nuevamente los valores faltantes.
final_missing_count = clean_df.isna().sum().sum()

# Contamos los duplicados que quedan.
final_duplicate_count = clean_df.duplicated().sum()

# Mostramos el resumen final.
print("Dimensiones finales:", clean_df.shape)
print("Total de valores faltantes:", final_missing_count)
print("Total de duplicados:", final_duplicate_count)

print("\nTipos de datos finales:")
print(clean_df.dtypes)

print("\nColumnas finales:")
print(clean_df.columns.tolist())

Dimensiones finales: (1500, 6)
Total de valores faltantes: 0
Total de duplicados: 0

Tipos de datos finales:
ID_Cliente       object
Ciudad           object
Vendedor         object
Categoria        object
Edad_Cliente    float64
Monto_Venta     float64
dtype: object

Columnas finales:
['ID_Cliente', 'Ciudad', 'Vendedor', 'Categoria', 'Edad_Cliente', 'Monto_Venta']


In [19]:
# Definimos las columnas que esperamos conservar después de la limpieza.
expected_columns = [
    "ID_Cliente",
    "Ciudad",
    "Vendedor",
    "Categoria",
    "Edad_Cliente",
    "Monto_Venta"
]

# Verificamos que no existan valores faltantes.
assert clean_df.isna().sum().sum() == 0

# Verificamos que no existan duplicados.
assert clean_df.duplicated().sum() == 0

# Verificamos que las columnas sean las esperadas.
assert clean_df.columns.tolist() == expected_columns

print("Todas las validaciones fueron aprobadas correctamente.")

Todas las validaciones fueron aprobadas correctamente.


Utilizamos instrucciones `assert` como controles automáticos.

Si alguna condición no se cumple, Colab mostraría un error. Como todas las validaciones fueron aprobadas, podemos afirmar que el DataFrame cumple.

In [20]:
# Creamos una tabla para comparar el estado original con el conjunto limpio.
comparison_summary = pd.DataFrame({
    "metric": [
        "rows",
        "columns",
        "total_missing_values",
        "duplicate_rows"
    ],
    "before_cleaning": [
        original_df.shape[0],
        original_df.shape[1],
        original_df.isna().sum().sum(),
        original_df.duplicated().sum()
    ],
    "after_cleaning": [
        clean_df.shape[0],
        clean_df.shape[1],
        clean_df.isna().sum().sum(),
        clean_df.duplicated().sum()
    ]
})

comparison_summary

,metric,before_cleaning,after_cleaning
0,rows,1590,1500
1,columns,8,6
2,total_missing_values,1866,0
3,duplicate_rows,90,0


Esta tabla resume el efecto del proceso realizado.

Inicialmente teníamos 1.590 filas, 8 columnas, 1.866 valores faltantes y 90 filas duplicadas.

Después de la limpieza obtuvimos 1.500 filas, 6 columnas, ningún valor faltante y ningún duplicado.

In [21]:
# Mostramos los primeros registros del DataFrame limpio.
clean_df.head(10)

,ID_Cliente,Ciudad,Vendedor,Categoria,Edad_Cliente,Monto_Venta
0,CLI-2079,Cali,Laura Torres,Ropa,56.0,8575896.0
1,CLI-1405,Cali,Laura Torres,Ropa,23.0,9635301.0
2,CLI-2493,Bogotá,Carlos Ruiz,Ropa,60.0,73060.0
3,CLI-1239,Medellín,Carlos Ruiz,Electrónica,36.0,232758.0
4,CLI-1610,Cali,Laura Torres,Hogar,26.0,677935.0
5,CLI-1099,Barranquilla,Laura Torres,Hogar,61.0,5184986.0
6,CLI-2145,Barranquilla,Laura Torres,Deportes,21.0,7604525.0
7,CLI-1297,Barranquilla,Carlos Ruiz,Hogar,58.0,4062451.0
8,CLI-1932,Medellín,María Gómez,Deportes,51.0,5885467.0
9,CLI-1275,Bogotá,Laura Torres,Hogar,20.0,4834993.0


Finalmente observamos nuevamente algunos registros para confirmar que la información conserva una estructura clara y que las columnas importantes siguen disponibles.

Este DataFrame ya puede utilizarse en análisis exploratorio, visualizaciones o procesos posteriores de aprendizaje computacional.

## Exportación del conjunto de datos limpio

Guardamos el resultado en un nuevo archivo CSV. No sobrescribimos el archivo original porque es importante conservar la fuente inicial y generar un archivo separado con los datos depurados.

In [22]:
# Definimos el nombre del archivo limpio.
clean_file_name = "clean_sales_data.csv"

# Guardamos el DataFrame sin incluir el índice de Pandas.
clean_df.to_csv(
    clean_file_name,
    index=False
)

print(f"Archivo guardado correctamente como: {clean_file_name}")

Archivo guardado correctamente como: clean_sales_data.csv


In [23]:
# Descargamos el archivo limpio desde Google Colab.
files.download(clean_file_name)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Conclusiones

Como equipo podemos concluir que el conjunto de datos inicial presentaba varios problemas de calidad:

- Existían 1.866 valores faltantes.
- Se encontraron 90 filas completamente duplicadas.
- La columna `Pais` no aportaba variación porque todos los registros pertenecían a Colombia.
- La columna `Comentarios_Cliente` tenía un porcentaje muy alto de valores faltantes.
- Las columnas `Edad_Cliente` y `Monto_Venta` fueron completadas utilizando la mediana.
- La columna `Categoria` fue completada utilizando la moda.

Después del proceso obtuvimos un conjunto de datos con 1.500 filas y 6 columnas. El archivo final no contiene valores faltantes ni filas duplicadas.

Las decisiones fueron tomadas buscando conservar la mayor cantidad de información útil, evitando eliminar registros válidos sin necesidad. El conjunto limpio queda preparado para continuar con análisis estadístico o con un modelo de aprendizaje computacional.

## Declaración sobre el uso de inteligencia artificial

Durante la elaboración de este taller, como equipo utilizamos una herramienta de inteligencia artificial como apoyo académico y técnico. La herramienta nos ayudó principalmente a organizar el notebook, revisar algunas decisiones de limpieza y mejorar la explicación de los procedimientos.

La inteligencia artificial no fue utilizada para reemplazar nuestro análisis. Nosotros cargamos el archivo, ejecutamos el código en Google Colab, revisamos los resultados y tomamos las decisiones relacionadas con las columnas, los valores faltantes y los registros duplicados.

### Prompts utilizados como apoyo

A continuación presentamos algunos de los prompts que utilizamos o adaptamos durante el desarrollo del trabajo:

1. **Organización general del taller**

   > Actúa como un ingeniero de sistemas experto en aprendizaje computacional y ayúdanos a organizar un taller de limpieza de datos con Pandas en Google Colab. El trabajo debe estar dividido paso a paso y cada bloque debe incluir una explicación clara de su objetivo.

   **Propósito:** Nos ayudó a organizar el notebook en diagnóstico inicial, selección de columnas, tratamiento de valores faltantes, eliminación de duplicados y validación final.

2. **Análisis de valores faltantes**

   > Explícanos cómo calcular en Pandas la cantidad y el porcentaje de valores faltantes por columna, utilizando un lenguaje sencillo y adecuado para un informe académico realizado por un equipo.

   **Propósito:** Nos permitió complementar el conteo de valores nulos con su porcentaje, para entender mejor qué columnas estaban más afectadas.

3. **Decisión entre media, mediana y moda**

   > Tenemos un conjunto de datos de ventas con columnas numéricas y categóricas que contienen valores faltantes. ¿Cuándo es mejor utilizar la mediana y cuándo utilizar la moda? Explícalo con un ejemplo sencillo.

   **Propósito:** Nos ayudó a justificar el uso de la mediana para `Edad_Cliente` y `Monto_Venta`, y de la moda para `Categoria`.

4. **Análisis de columnas poco útiles**

   > ¿Cómo se puede determinar si una columna debe eliminarse porque todos sus registros tienen el mismo valor o porque contiene demasiados datos faltantes?

   **Propósito:** Con este apoyo entendimos que `Pais` no aportaba variación y que `Comentarios_Cliente` tenía demasiados valores faltantes para conservarla de forma confiable.

5. **Tratamiento de filas duplicadas**

   > Explícanos cómo identificar filas completamente duplicadas en Pandas y cómo conservar únicamente la primera aparición de cada registro. Incluye la forma de verificar el resultado.

   **Propósito:** Nos orientó en el uso de `duplicated()` y `drop_duplicates(keep="first")`.

6. **Buenas prácticas de programación**

   > Revisa la estructura de un código de limpieza de datos en Pandas y propón buenas prácticas de nomenclatura en inglés, comentarios en español y separación del código por etapas.

   **Propósito:** Nos ayudó a utilizar nombres de variables en inglés, conservar comentarios claros en español y organizar el código para que fuera más fácil de leer y revisar.

7. **Validación final del proceso**

   > ¿Qué validaciones se deben realizar después de limpiar un DataFrame para confirmar que no quedan valores nulos, duplicados o columnas incorrectas?

   **Propósito:** Nos permitió incluir una comprobación final de dimensiones, valores faltantes, duplicados, nombres de columnas y tipos de datos.

### Forma en que utilizamos la herramienta

La herramienta fue utilizada como apoyo para comprender temas que no estaban completamente desarrollados en clase, especialmente el cálculo de porcentajes de datos faltantes, la diferencia entre mediana y moda, la decisión de eliminar columnas con poca información y la creación de validaciones automáticas.

El código fue revisado, ajustado y ejecutado por los integrantes del equipo. También verificamos que los resultados obtenidos coincidieran con las características del archivo entregado.

Por lo tanto, asumimos la responsabilidad sobre el análisis, las decisiones tomadas, la ejecución del notebook y las conclusiones presentadas en este taller.